# E0 — Readout Validity Across Depth

**Plan reference:** Phase 0, experiment E0 (with E0b, the norm-normalisation decision, folded in as §4).

## The question

Nearly every later experiment in this plan works by taking a residual-stream vector from some
*middle* layer of the model and projecting it onto a fixed direction built from the model's
**output** weights — for example the direction that separates the wrong answer `25` from the
correct answer `27`:

$$w = W_U[:, \text{“25”}] - W_U[:, \text{“27”}]$$

This is the standard *logit lens*. It carries a hidden assumption:

> **that a direction which means "prefers 25 over 27" at the final layer means the same thing at
> layer 3, layer 6, layer 9…**

That assumption is not obviously true. Representations can rotate across depth, and LayerNorm
rescales things layer by layer. If the assumption fails at exactly the depths where the "impulse"
supposedly lives, then every heatmap produced by E2 and every persistence claim in E3 is
untrustworthy *before anyone interprets it*.

**So E0 is not a test of the momentum theory.** It is a test of the instrument. Its output is a
depth range within which fixed-direction projections can be believed — a precondition for reading
anything else.

## What this notebook does

1. Builds the W-vs-C direction and reads it out with the plain **logit lens** at every layer.
2. Checks how much of any apparent trend is just **residual-stream norm growth** (E0b).
3. Fits a per-layer **least-squares translator** — a cheap stand-in for a *tuned lens* — that maps
   layer-$\ell$ activations into final-layer coordinates, and re-reads the same direction through it.
4. **Compares** the two readouts and reports where they agree and where they diverge.
5. Runs a **norm-matched random-direction control**, so that "the two lenses disagree" can be
   distinguished from "the two lenses disagree *about this direction specifically*".
6. Prints a recommended trustworthy depth range.

## How to read a result

- **The two readouts largely agree from some layer onwards** → the logit lens is usable in that
  range, and E2/E3 can proceed with the plain, cheap method inside it.
- **They diverge badly in early/middle layers** → any fixed-direction result at those depths needs
  the corrected readout, or needs to be discarded. This is the outcome the J-lens paper's findings
  would lead you to expect.
- **They diverge everywhere, including late layers** → something is wrong with the setup itself
  (fit, normalisation, tokenisation), not with the lens. Debug before concluding anything.

## Honest caveats, stated up front

- The translator fitted here is an **approximation** of a tuned lens. A real tuned lens is trained
  to minimise KL divergence to the final output distribution; this one is fitted by least squares
  to predict the final residual vector. It is cheaper, needs no training loop, and is adequate for
  a first "is the lens trustworthy" check — but it is not the published method. For anything
  load-bearing, use the `tuned-lens` package or the Jacobian-lens approach.
- Everything below runs on **one prompt** for the direction-specific part. That answers "is the
  instrument trustworthy here", not "is it trustworthy in general". The fragility threshold
  (how many prompts before believing a pattern) still needs deciding, and should be decided
  *before* looking at output.
- **Requires downloading GPT-2 weights**, so it needs a network connection on first run.

In [ ]:
# --- Install (uncomment on a fresh environment) ---------------------------------
# %pip install transformer-lens torch matplotlib numpy

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

torch.set_grad_enabled(False)   # we never train anything here; this saves memory

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Load the model

`from_pretrained` applies TransformerLens's standard preprocessing (folding LayerNorm weights into
the following linear layers, centering the unembedding). This does not change the model's function
— identical logits — it just makes the internals easier to reason about. `model.ln_final` still
exists and still performs the actual normalisation, which matters below.

In [ ]:
model = HookedTransformer.from_pretrained("gpt2-small", device=DEVICE)

N_LAYERS = model.cfg.n_layers      # 12 for gpt2-small
D_MODEL  = model.cfg.d_model       # 768
print(f"layers: {N_LAYERS}   d_model: {D_MODEL}")

# NOTE ON RESOLUTION, carried over from the plan:
# 12 layers means every "across depth" curve below has ~13 points. That is enough to see
# "rises once" vs "rises and falls". It is NOT enough to claim periodicity or spectral structure.

## 2. The prompt and the direction

The working example from the plan. `W` is the wrong answer the model has been made to commit to;
`C` is the correct one. The direction `w` is their difference in unembedding space.

**Why the difference, not just `W_U[:, W]`?** Using the W column alone would also pick up anything
that raises *all* logits together (general confidence or scale effects), which has nothing to do
with the 25-vs-27 choice. Subtracting cancels that shared component, leaving only the axis along
which the two answers pull against each other.

In [ ]:
PROMPT = "12 + 15 = 25. Wait, let me recompute. 12 + 15 ="

tokens     = model.to_tokens(PROMPT)             # [1, seq]
str_tokens = model.to_str_tokens(PROMPT)
print("tokenisation:")
for i, t in enumerate(str_tokens):
    print(f"  {i:2d}  {t!r}")

# --- The two answer tokens ------------------------------------------------------
# GPT-2's tokeniser is whitespace-sensitive: " 25" and "25" are different tokens.
# The model would emit " 25" after "=", so we want the leading-space versions.
# to_single_token raises if the string is not exactly one token — that assertion is the point.
W_STR, C_STR = " 25", " 27"
W_tok = model.to_single_token(W_STR)    # wrong answer
C_tok = model.to_single_token(C_STR)    # correct answer
print(f"\nW = {W_STR!r} -> id {W_tok}      C = {C_STR!r} -> id {C_tok}")

# --- The direction --------------------------------------------------------------
# W_U has shape [d_model, d_vocab]; each column is the direction that raises one token's logit.
w_dir = model.W_U[:, W_tok] - model.W_U[:, C_tok]     # [d_model]
print("direction shape:", tuple(w_dir.shape), " norm:", float(w_dir.norm()))

In [ ]:
# --- Locate the positions we care about ----------------------------------------
# Written as a search rather than hardcoded indices so that changing PROMPT does not
# silently produce wrong results.

def find_token_indices(str_tokens, target):
    return [i for i, t in enumerate(str_tokens) if t == target]

impulse_positions = find_token_indices(str_tokens, " 25")   # where the wrong answer was asserted
eq_positions      = find_token_indices(str_tokens, " =")

IMPULSE_POS = impulse_positions[0]      # the first (and here only) " 25"
T_STAR      = eq_positions[-1]          # final "=" — the probe position, where the model would
                                        # emit its post-revision answer

print(f"impulse position (first ' 25'): {IMPULSE_POS}  -> {str_tokens[IMPULSE_POS]!r}")
print(f"probe position t* (final ' ='): {T_STAR}  -> {str_tokens[T_STAR]!r}")

## 3. Readout A — the plain logit lens

For each layer we take `resid_post` (the residual stream *after* that layer has written to it),
push it through the model's final LayerNorm, and dot it with `w`.

**The LayerNorm step is not optional.** The unembedding was trained to read normalised vectors. If
you skip `ln_final`, the numbers you get are dominated by how large the residual happens to be at
that depth, not by which direction it points. Both versions are computed below precisely so the
difference is visible rather than assumed.

Layer index 0 here means "the embedding, before any transformer block" — so there are `n_layers + 1`
points.

In [ ]:
_, cache = model.run_with_cache(tokens)

def residual_stack(cache):
    # Returns [n_layers+1, seq, d_model]: the embedding, then resid_post after each layer.
    stack = [cache["resid_pre", 0][0]]                          # embeddings + positional
    for l in range(N_LAYERS):
        stack.append(cache["resid_post", l][0])
    return torch.stack(stack, dim=0)

resid = residual_stack(cache)                # [L+1, seq, d_model]
print("residual stack:", tuple(resid.shape))

# With ln_final applied (the correct logit lens) ...
resid_ln = model.ln_final(resid)             # LayerNorm broadcasts over leading dims
proj_ln  = (resid_ln @ w_dir).cpu().numpy()  # [L+1, seq]

# ... and without, purely for the comparison in the next section.
proj_raw = (resid @ w_dir).cpu().numpy()     # [L+1, seq]

print(f"\nlogit-lens W-vs-C projection at probe position t*={T_STAR}, by layer:")
for l in range(N_LAYERS + 1):
    label = "embed" if l == 0 else f"L{l-1}"
    print(f"  {label:>6}  {proj_ln[l, T_STAR]:+8.3f}")

In [ ]:
# --- Plot the two positions that matter, with and without LayerNorm -------------

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
xs = np.arange(N_LAYERS + 1)

for ax, data, title in [
    (axes[0], proj_ln,  "WITH ln_final  (the correct logit lens)"),
    (axes[1], proj_raw, "WITHOUT ln_final  (for contrast only)"),
]:
    ax.plot(xs, data[:, T_STAR],      marker="o", label=f"probe t* (pos {T_STAR})")
    ax.plot(xs, data[:, IMPULSE_POS], marker="s", label=f"impulse (pos {IMPULSE_POS})")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("layer (0 = embedding)")
    ax.set_ylabel("projection onto W - C")
    ax.legend()

plt.tight_layout(); plt.show()

# Positive = leans toward the wrong answer 25. Negative = leans toward 27.
# If the two panels tell noticeably different stories, that is E0b already answering itself.

## 4. E0b — how much of this is just norm growth?

The residual stream typically grows in magnitude with depth. That alone will make raw dot products
trend upward, producing an apparent "signal builds across layers" that is really an artifact of
scale.

This section makes that concrete: plot the norms, then check how much of the raw projection is
explained by norm alone. If the raw curve tracks the norm curve closely, the raw readout is
measuring size, not direction — and every depth comparison from here on must use the
LayerNorm-applied (or otherwise normalised) version.

In [ ]:
norms = resid.norm(dim=-1).cpu().numpy()      # [L+1, seq]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(xs, norms[:, T_STAR],      marker="o", label="probe t*")
axes[0].plot(xs, norms[:, IMPULSE_POS], marker="s", label="impulse")
axes[0].set_title("Residual-stream norm by layer")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("||resid||"); axes[0].legend()

# Cosine similarity strips magnitude out entirely: pure direction.
cos_sim = (resid @ w_dir / (resid.norm(dim=-1) * w_dir.norm())).cpu().numpy()
axes[1].plot(xs, cos_sim[:, T_STAR],      marker="o", label="probe t*")
axes[1].plot(xs, cos_sim[:, IMPULSE_POS], marker="s", label="impulse")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Cosine similarity to (W - C)  — magnitude removed")
axes[1].set_xlabel("layer"); axes[1].set_ylabel("cosine"); axes[1].legend()

plt.tight_layout(); plt.show()

# How much of the RAW projection is explained by norm growth alone?
r = np.corrcoef(norms[:, T_STAR], proj_raw[:, T_STAR])[0, 1]
print(f"corr(norm, raw projection) at t*: {r:+.3f}")
print("A large positive value means the un-normalised readout is largely tracking size, not direction.")

## 5. Readout B — a fitted translator (stand-in for a tuned lens)

The logit lens assumes layer-$\ell$ coordinates are already final-layer coordinates. The fix is to
stop assuming and instead **measure** the map between them.

For each layer $\ell$ we fit an affine map

$$h_{\text{final}} \approx A_\ell \, h_\ell + b_\ell$$

by ridge regression over many token positions drawn from ordinary text. Then we read the direction
through that map instead of directly.

**What this is and is not.** A published tuned lens is trained to minimise KL divergence to the
final output distribution. This is a cheaper least-squares fit to the final *residual vector*. It
is enough to answer "does the naive lens mislead me here, and where", which is all E0 needs. It is
not a substitute for the real method in anything load-bearing.

**Watch the printed sample count.** We are fitting a 768×768 matrix per layer. If the number of
token positions is not comfortably larger than 768, the fit is underdetermined and the ridge
penalty is doing most of the work — the comparison then says more about the regulariser than about
the model. Add more text if `N` comes out small.

In [ ]:
# A small corpus, hardcoded so this notebook needs no dataset download.
# Replace with real corpus text for anything serious — this is deliberately minimal.
FIT_TEXTS = [
    "The city council met on Tuesday to discuss the proposed changes to the zoning laws.",
    "Water boils at one hundred degrees Celsius at standard atmospheric pressure.",
    "She opened the letter carefully, unsure whether she wanted to read what was inside.",
    "In the early nineteenth century, railways transformed the movement of goods across Europe.",
    "The function returns a list of integers sorted in ascending order by default.",
    "Photosynthesis converts light energy into chemical energy stored in glucose molecules.",
    "He argued that the evidence was circumstantial and did not support the conclusion.",
    "Add the flour gradually, stirring constantly until the mixture thickens.",
    "The orbit of the planet is elliptical rather than perfectly circular.",
    "Most of the delegates arrived the night before the conference began.",
    "A standing wave forms when two waves of equal frequency travel in opposite directions.",
    "The report was published without any mention of the earlier findings.",
    "Children learn language rapidly during the first few years of life.",
    "Copper conducts electricity far better than iron does under the same conditions.",
    "They walked along the river until the path disappeared into the trees.",
    "The committee postponed the vote until further information became available.",
    "Sedimentary rock forms from particles deposited over long periods of time.",
    "Every element in the array is checked exactly once during a linear scan.",
]

# Collect (layer activations, final activations) pairs across all these texts.
per_layer_X = [[] for _ in range(N_LAYERS + 1)]
Y_final     = []

for text in FIT_TEXTS:
    toks = model.to_tokens(text)
    _, c = model.run_with_cache(toks)
    stack = residual_stack(c)                      # [L+1, seq, d_model]
    for l in range(N_LAYERS + 1):
        per_layer_X[l].append(stack[l])
    Y_final.append(stack[-1])                      # target = final resid_post

per_layer_X = [torch.cat(v, dim=0).float() for v in per_layer_X]   # each [N, d_model]
Y_final     = torch.cat(Y_final, dim=0).float()                    # [N, d_model]

N = Y_final.shape[0]
print(f"fitting on N = {N} token positions,  d_model = {D_MODEL}")
if N < 3 * D_MODEL:
    print("  WARNING: N is small relative to d_model. The ridge penalty will dominate the fit.")
    print("           Add more/longer texts to FIT_TEXTS before trusting the comparison.")

In [ ]:
def fit_affine(X, Y, ridge=1.0):
    # Least-squares fit of Y ~ X @ A + b, solved in closed form with an L2 penalty.
    # A bias column of ones is appended so the intercept is fitted alongside A.
    # Returns A [d, d], b [d].
    ones = torch.ones(X.shape[0], 1, device=X.device, dtype=X.dtype)
    Xa   = torch.cat([X, ones], dim=1)                       # [N, d+1]
    G    = Xa.T @ Xa                                         # [d+1, d+1]
    G   += ridge * torch.eye(G.shape[0], device=G.device, dtype=G.dtype)
    Sol  = torch.linalg.solve(G, Xa.T @ Y)                   # [d+1, d]
    return Sol[:-1, :], Sol[-1, :]

translators = []
for l in range(N_LAYERS + 1):
    A, b = fit_affine(per_layer_X[l], Y_final, ridge=1.0)
    translators.append((A, b))
    # R^2 tells us how well the affine map actually captures this layer -> final relationship.
    pred  = per_layer_X[l] @ A + b
    ss_res = ((Y_final - pred) ** 2).sum()
    ss_tot = ((Y_final - Y_final.mean(0)) ** 2).sum()
    label = "embed" if l == 0 else f"L{l-1}"
    print(f"  {label:>6}  fit R^2 = {1 - ss_res/ss_tot:.3f}")

print("\nLow R^2 at a layer means the layer->final relationship is poorly captured by ANY affine map.")
print("At those depths, both lenses are suspect — not just the naive one.")

## 6. Comparison — where do the two readouts agree?

Now read the same direction both ways on the test prompt and compare.

Three statistics, each answering something slightly different:

- **Correlation across positions** — do the two lenses rank the sequence positions the same way at
  this depth? This is the most direct "do they mean the same thing" measure.
- **Sign agreement** — do they at least agree on *which answer is favoured*? A weaker but more
  interpretable bar: a lens can be miscalibrated in magnitude and still be usable for direction.
- **Emergence layer** — the first layer at which each lens says W-preference appears and persists.
  If the two disagree about this, they disagree about exactly the thing E2 is trying to find.

In [ ]:
# Translate every layer's activations into final-layer coordinates, then read them.
proj_tuned = np.zeros((N_LAYERS + 1, resid.shape[1]))
for l in range(N_LAYERS + 1):
    A, b = translators[l]
    translated = resid[l].float() @ A + b            # [seq, d_model], now in final coordinates
    proj_tuned[l] = (model.ln_final(translated) @ w_dir).cpu().numpy()

# --- Statistic 1: correlation across sequence positions, per layer --------------
corrs = np.array([np.corrcoef(proj_ln[l], proj_tuned[l])[0, 1] for l in range(N_LAYERS + 1)])

# --- Statistic 2: sign agreement across positions, per layer --------------------
sign_agree = np.array([np.mean(np.sign(proj_ln[l]) == np.sign(proj_tuned[l]))
                       for l in range(N_LAYERS + 1)])

print(f"{'layer':>6}  {'corr':>7}  {'sign agree':>11}")
for l in range(N_LAYERS + 1):
    label = "embed" if l == 0 else f"L{l-1}"
    print(f"{label:>6}  {corrs[l]:+7.3f}  {sign_agree[l]:11.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(xs, proj_ln[:, T_STAR],    marker="o", label="logit lens")
axes[0].plot(xs, proj_tuned[:, T_STAR], marker="s", label="fitted translator")
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title(f"W-vs-C readout at probe t* (pos {T_STAR})")
axes[0].set_xlabel("layer"); axes[0].set_ylabel("projection"); axes[0].legend()

axes[1].plot(xs, corrs,      marker="o", label="corr across positions")
axes[1].plot(xs, sign_agree, marker="s", label="sign agreement")
axes[1].axhline(0.9, color="red", ls="--", lw=0.8, label="0.9 reference")
axes[1].set_ylim(-1.05, 1.05)
axes[1].set_title("Agreement between the two lenses, by layer")
axes[1].set_xlabel("layer"); axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
# --- Statistic 3: emergence layer -----------------------------------------------
# First layer at which the W-preference at t* turns positive AND stays positive.

def emergence_layer(series):
    for l in range(len(series)):
        if np.all(series[l:] > 0):
            return l
    return None

e_ln    = emergence_layer(proj_ln[:, T_STAR])
e_tuned = emergence_layer(proj_tuned[:, T_STAR])

def fmt(l):
    if l is None: return "never"
    return "embed" if l == 0 else f"L{l-1}"

print(f"logit lens says W-preference emerges at:        {fmt(e_ln)}")
print(f"fitted translator says it emerges at:           {fmt(e_tuned)}")
if e_ln is not None and e_tuned is not None and e_ln != e_tuned:
    print(f"\n  -> The two lenses DISAGREE by {abs(e_ln - e_tuned)} layer(s) about where commitment appears.")
    print("     E2 is looking for exactly this quantity, so this disagreement is the finding.")

## 7. Control — norm-matched random directions

Some disagreement between two different readouts is expected simply because they are different
readouts. The question is whether the W-vs-C direction disagrees *more* than an arbitrary direction
would.

So: sample random directions with the same norm as `w`, run the identical comparison, and use their
distribution as the null. If W-vs-C sits inside that distribution, the disagreement observed above
is generic lens behaviour rather than anything about this direction.

This is the control the plan flags as applying to everything — worth running here, at the cheapest
possible experiment, to establish the habit.

In [ ]:
N_RANDOM = 30
rng = torch.Generator(device=DEVICE).manual_seed(0)   # fixed seed: this control is reproducible

random_corrs = np.zeros((N_RANDOM, N_LAYERS + 1))
for i in range(N_RANDOM):
    r_dir = torch.randn(D_MODEL, generator=rng, device=DEVICE, dtype=w_dir.dtype)
    r_dir = r_dir / r_dir.norm() * w_dir.norm()       # norm-matched to w

    p_ln = (resid_ln @ r_dir).cpu().numpy()
    p_tu = np.zeros_like(p_ln)
    for l in range(N_LAYERS + 1):
        A, b = translators[l]
        p_tu[l] = (model.ln_final(resid[l].float() @ A + b) @ r_dir).cpu().numpy()

    random_corrs[i] = [np.corrcoef(p_ln[l], p_tu[l])[0, 1] for l in range(N_LAYERS + 1)]

lo, med, hi = np.percentile(random_corrs, [5, 50, 95], axis=0)

plt.figure(figsize=(7, 4))
plt.fill_between(xs, lo, hi, alpha=0.25, label="random directions (5-95%)")
plt.plot(xs, med, ls="--", label="random median")
plt.plot(xs, corrs, marker="o", color="crimson", label="W - C direction")
plt.axhline(0, color="black", lw=0.8)
plt.title("Is W-vs-C harder to read than an arbitrary direction?")
plt.xlabel("layer"); plt.ylabel("lens agreement (corr)"); plt.legend()
plt.tight_layout(); plt.show()

inside = (corrs >= lo) & (corrs <= hi)
print("Layers where W-vs-C agreement sits INSIDE the random band (i.e. unremarkable):")
print("  " + ", ".join("embed" if l == 0 else f"L{l-1}" for l in range(N_LAYERS + 1) if inside[l]))

## 8. The deliverable — a trustworthy depth range

E0's output is meant to be a single practical statement: *within this range of layers, a fixed
`W_U`-derived direction can be believed; outside it, use the corrected readout or discard.*

**The thresholds below are arbitrary.** They should be set before looking at the plots and then
left alone — otherwise this becomes an exercise in choosing a cutoff that produces a congenial
answer, which is the pre-registration failure mode the plan warns about for every experiment.

In [ ]:
# --- Pre-register these BEFORE reading any output above -------------------------
CORR_THRESHOLD = 0.90   # lenses must rank positions near-identically
SIGN_THRESHOLD = 0.90   # and agree on which answer is favoured at 90%+ of positions
R2_THRESHOLD   = 0.50   # and the affine layer->final map must be a decent fit at all

r2s = []
for l in range(N_LAYERS + 1):
    A, b = translators[l]
    pred   = per_layer_X[l] @ A + b
    ss_res = ((Y_final - pred) ** 2).sum()
    ss_tot = ((Y_final - Y_final.mean(0)) ** 2).sum()
    r2s.append(float(1 - ss_res / ss_tot))
r2s = np.array(r2s)

trustworthy = (corrs >= CORR_THRESHOLD) & (sign_agree >= SIGN_THRESHOLD) & (r2s >= R2_THRESHOLD)

print("Per-layer verdict:")
for l in range(N_LAYERS + 1):
    label = "embed" if l == 0 else f"L{l-1}"
    mark  = "OK  " if trustworthy[l] else "SUSPECT"
    print(f"  {label:>6}  {mark}   corr={corrs[l]:+.3f}  sign={sign_agree[l]:.2f}  R2={r2s[l]:.3f}")

ok_layers = [l for l in range(N_LAYERS + 1) if trustworthy[l]]
print()
if ok_layers:
    print(f"TRUSTWORTHY RANGE: layers {fmt(min(ok_layers))} through {fmt(max(ok_layers))}")
    print("E2 / E3 results should be read inside this range, or use the translated readout outside it.")
else:
    print("NO layer clears all three thresholds.")
    print("Before concluding the lens is unusable, check the fit first: small N, or low R2 across")
    print("the board, points at the setup rather than at the model.")

## 9. What this does and does not establish

**Establishes (for this prompt, this model, this direction):** a depth range within which the cheap
fixed-direction readout can be believed, and a control showing whether any disagreement is specific
to W-vs-C or generic to the lens.

**Does not establish:**

- *Anything about momentum.* E0 is instrument calibration. No result here bears on whether
  commitment persists, whether there is an impulse, or whether anything standing-wave-like exists.
- *Generality.* One prompt. The fragility threshold — how many prompts, how much variation in
  operands and cue phrasing, before a pattern is believed — is still undecided and should be fixed
  before running E2.
- *That the translated readout is correct.* It is a least-squares approximation of a tuned lens.
  It is a second opinion, not ground truth. If the two lenses disagree, that means one of them is
  wrong somewhere, not that the fitted one is right.

**Natural next steps:**

- If a trustworthy range exists → run **E2** (write localisation) restricted to it.
- If the trustworthy range excludes the depths where an impulse would plausibly live → adopt a
  proper tuned lens or the Jacobian-lens approach before proceeding, rather than reading heatmaps
  through a lens already known to mislead at those depths.
- Independently of the outcome → **E1** (behavioural revision curve) and **E6** (operand corruption)
  need none of this machinery and can be run in parallel.